<a href="https://colab.research.google.com/github/marialabadi/big-data/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin

print("Libraries imported successfully!")

Libraries imported successfully!


In [49]:
# تحديد الموقع الأساسي للموقع وإنشاء User-Agent للتعريف بالطلب ثم طباعة الرابط
BASE_URL = "https://en.wikiquote.org"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; TP01-Web-Scraping/1.0)"
}

print(BASE_URL)

https://en.wikiquote.org


In [50]:
# تحديد رابط صفحة جميع الصفحات
url = BASE_URL + "/wiki/Special:AllPages"
# إرسال طلب لجلب محتوى الصفحة
response = requests.get(
    url,
    headers=headers,
    timeout=20
)

print("Status code:", response.status_code)
print("HTML length:", len(response.text))

Status code: 200
HTML length: 66123


In [51]:
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.get_text(strip=True))

Page title: All pages - Wikiquote


In [52]:
print(response.text[:5000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-disabled vector-feature-custom-font-size-clientpref--excluded vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-not-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>All pages - Wikiquote</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabl

In [53]:
# استخراج جميع الروابط الموجودة في صفحة HTML
links = soup.find_all("a", href=True)

print("Number of links:", len(links))

Number of links: 393


In [54]:
# إنشاء قائمة لتخزين روابط صفحات Wikiquote
page_links = []
# المرور على جميع الروابط التي تم العثور عليها
for link in links:
    href = link.get("href")

    if href and href.startswith("/wiki/"):
        if not href.startswith("/wiki/Special:"):
            page_links.append(urljoin(BASE_URL, href))
# حذف الروابط المكررة
page_links = list(dict.fromkeys(page_links))

print("Wikiquote page links found:", len(page_links))
print(page_links[:10])

Wikiquote page links found: 359
['https://en.wikiquote.org/wiki/Main_Page', 'https://en.wikiquote.org/wiki/Wikiquote:Community_Portal', 'https://en.wikiquote.org/wiki/Wikiquote:Village_pump', 'https://en.wikiquote.org/wiki/Help:Contents', 'https://en.wikiquote.org/wiki/Wikiquote:Contact_us', 'https://en.wikiquote.org/wiki/List_of_people_by_name', 'https://en.wikiquote.org/wiki/List_of_literary_works', 'https://en.wikiquote.org/wiki/Category:Proverbs', 'https://en.wikiquote.org/wiki/List_of_films', 'https://en.wikiquote.org/wiki/List_of_television_shows']


In [56]:
# تحديد صفحة لاختبار استخراج البيانات منها
test_url = BASE_URL + "/wiki/Albert_Einstein"
# إرسال طلب لجلب صفحة Albert Einstein
test_response = requests.get(
    test_url,
    headers=headers,
    timeout=20
)
# التأكد من نجاح تحميل الصفحة
print("Status:", test_response.status_code)

Status: 200


In [57]:
# تحويل كود HTML الخاص بالصفحة إلى كائن BeautifulSoup لتحليله
test_soup = BeautifulSoup(test_response.text, "html.parser")

print("Title:", test_soup.title.get_text(strip=True))

Title: Albert Einstein - Wikiquote


In [58]:
# البحث عن جميع الاقتباسات الموجودة داخل وسم blockquote
quotes = test_soup.find_all("blockquote")
# عرض عدد الاقتباسات التي تم العثور عليها
print("Quotes found:", len(quotes))

for q in quotes:
    print(q.get_text(" ", strip=True))

Quotes found: 2
Joe Laitin reports that reporters at Bikini were questioning an army lieutenant about what weapons would be used in the next war. "I dunno," he said, "but in the war after the next war, sure as Hell, they'll be using spears!"
The wording of the statement you have quoted is not my own. Shortly after Hitler came to power in Germany I had an oral conversation with a newspaper man about these matters. Since then my remarks have been elaborated and exaggerated nearly beyond recognition. I cannot in good conscience write down the statement you sent me as my own. The matter is all the more embarrassing to me because I, like yourself, I am predominantly critical concerning the activities, and especially the political activities, through history of the official clergy. Thus, my former statement, even if reduced to my actual words (which I do not remember in detail) gives a wrong impression of my general attitude.


In [62]:
# إنشاء دالة لاستخراج الاقتباسات من صفحة معينة
def extract_quotes(url):
    try:
               # إرسال طلب لجلب محتوى الصفحة
               response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        if response.status_code != 200:
            return []
# تحويل HTML إلى BeautifulSoup لتحليل محتوى الصفحة
        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.get_text(" ", strip=True) if soup.title else ""
 # إنشاء قائمة لتخزين الاقتباسات المستخرجة
        results = []

         # البحث عن عناوين الأقسام في الصفحة
        headings = soup.find_all(["h2", "h3"])
# تحديد ما إذا كنا داخل قسم Quotes
        quotes_section = False
# المرور على العناوين وعناصر القوائم في الصفحة
        for element in soup.find_all(["h2", "h3", "li"]):

       # التحقق من وجود عنوان قسم Quotes
            if element.name in ["h2", "h3"]:
                heading_text = element.get_text(" ", strip=True).lower()
# بدء استخراج الاقتباسات بعد الوصول إلى قسم Quotes
                if "quotes" in heading_text:
                    quotes_section = True
                    continue

                # إيقاف استخراج الاقتباسات عند بداية قسم رئيسي جديد
                if quotes_section and element.name == "h2":
                    quotes_section = False
 # استخراج النصوص الموجودة داخل عناصر القائمة في قسم Quotes
            elif element.name == "li" and quotes_section:

                quote_text = element.get_text(" ", strip=True)
# إضافة الاقتباس إذا كان موجوداً وطوله أكثر من 20 حرفاً
                if quote_text and len(quote_text) > 20:
                    results.append({
                        "source_page": title,
                        "quote": quote_text,
                        "url": url
                    })
 # إرجاع جميع الاقتباسات المستخرجة
        return results
# التعامل مع الأخطاء أثناء تحميل أو تحليل الصفحة
    except Exception as e:
        print("Error:", url, e)
        return []

In [63]:
# استخراج الاقتباسات من صفحة الاختبار
test_quotes = extract_quotes(test_url)
# عرض عدد الاقتباسات التي تم استخراجها
print("Number of extracted quotes:", len(test_quotes))

for item in test_quotes:
    print(item)

Number of extracted quotes: 978
{'source_page': 'Albert Einstein - Wikiquote', 'quote': 'Everything should be made as simple as possible, but not simpler. Repeated throughout his life (with variants); see: Quote Investigator', 'url': 'https://en.wikiquote.org/wiki/Albert_Einstein'}
{'source_page': 'Albert Einstein - Wikiquote', 'quote': 'Repeated throughout his life (with variants); see: Quote Investigator', 'url': 'https://en.wikiquote.org/wiki/Albert_Einstein'}
{'source_page': 'Albert Einstein - Wikiquote', 'quote': 'Un homme heureux est trop content du présent pour trop se soucier de l\'avenir. A happy man is too satisfied with the present to dwell too much on the future. From "Mes Projets d\'Avenir", a French essay written at age 18 for a school exam (18 September 1896). The Collected Papers of Albert Einstein Vol. 1 (1987) Doc. 22.', 'url': 'https://en.wikiquote.org/wiki/Albert_Einstein'}
{'source_page': 'Albert Einstein - Wikiquote', 'quote': 'A happy man is too satisfied with th

In [64]:
# إنشاء قائمة لتخزين جميع الاقتباسات المستخرجة
all_quotes = []

# المرور على أول 100 صفحة من الموقع
for i, url in enumerate(page_links[:100], start=1):
   # عرض رقم الصفحة التي تتم معالجتها
    print(f"Processing page {i}/100")
    # استخراج الاقتباسات من الصفحة الحالية
    quotes = extract_quotes(url)
   # إضافة الاقتباسات المستخرجة إلى القائمة الرئيسية
    all_quotes.extend(quotes)

    time.sleep(0.5)

print("Total quotes collected:", len(all_quotes))

Processing page 1/100
Processing page 2/100
Processing page 3/100
Processing page 4/100
Processing page 5/100
Processing page 6/100
Processing page 7/100
Processing page 8/100
Processing page 9/100
Processing page 10/100
Processing page 11/100
Processing page 12/100
Processing page 13/100
Processing page 14/100
Processing page 15/100
Processing page 16/100
Processing page 17/100
Processing page 18/100
Processing page 19/100
Processing page 20/100
Processing page 21/100
Processing page 22/100
Processing page 23/100
Processing page 24/100
Processing page 25/100
Processing page 26/100
Processing page 27/100
Processing page 28/100
Processing page 29/100
Processing page 30/100
Processing page 31/100
Processing page 32/100
Processing page 33/100
Processing page 34/100
Processing page 35/100
Processing page 36/100
Processing page 37/100
Processing page 38/100
Processing page 39/100
Processing page 40/100
Processing page 41/100
Processing page 42/100
Processing page 43/100
Processing page 44/1

In [65]:
print("Current number of quotes:", len(all_quotes))

Current number of quotes: 924


In [67]:
all_quotes.extend(more_quotes)

print("Final number of quotes:", len(all_quotes))

Final number of quotes: 1258


In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# رابط موقع Wikiquote
BASE_URL = "https://en.wikiquote.org"

# إعداد User-Agent حتى يتعرف الموقع على الطلب
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; TP01-Web-Scraping/1.0)"
}

# تحميل صفحة AllPages
url = BASE_URL + "/wiki/Special:AllPages"

response = requests.get(
    url,
    headers=headers,
    timeout=20
)

# تحويل HTML إلى BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# استخراج جميع الروابط
links = soup.find_all("a", href=True)

# إنشاء قائمة روابط صفحات Wikiquote
page_links = []

for link in links:
    href = link.get("href")

    if href and href.startswith("/wiki/"):
        if not href.startswith("/wiki/Special:"):
            page_links.append(urljoin(BASE_URL, href))

# حذف الروابط المكررة
page_links = list(dict.fromkeys(page_links))

print("Number of page links:", len(page_links))
print("First 10 links:")
print(page_links[:10])

Number of page links: 359
First 10 links:
['https://en.wikiquote.org/wiki/Main_Page', 'https://en.wikiquote.org/wiki/Wikiquote:Community_Portal', 'https://en.wikiquote.org/wiki/Wikiquote:Village_pump', 'https://en.wikiquote.org/wiki/Help:Contents', 'https://en.wikiquote.org/wiki/Wikiquote:Contact_us', 'https://en.wikiquote.org/wiki/List_of_people_by_name', 'https://en.wikiquote.org/wiki/List_of_literary_works', 'https://en.wikiquote.org/wiki/Category:Proverbs', 'https://en.wikiquote.org/wiki/List_of_films', 'https://en.wikiquote.org/wiki/List_of_television_shows']


In [68]:
df = pd.DataFrame(all_quotes)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head(10)

Number of rows: 1258
Number of columns: 3


,source_page,quote,url
0,Ezra Klein - Wikiquote,If you are trying to create things that are go...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
1,Ezra Klein - Wikiquote,https://conversationswithtyler.com/episodes/ez...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
2,Ezra Klein - Wikiquote,There is no more profound human bias than the ...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
3,Ezra Klein - Wikiquote,"This Changes Everything (March 13, 2023), The ...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
4,Ezra Klein - Wikiquote,I have called the Republican Party an engine f...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
5,Ezra Klein - Wikiquote,Transcript: Ezra Klein Interviews Sam Rosenfel...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
6,Ezra Klein - Wikiquote,The Democratic Party does not need to choose t...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
7,Ezra Klein - Wikiquote,""" This Is the Way You Beat Trump — and Trumpis...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
8,"""Good chap"" theory - Wikiquote",The nature of its constitution tells you a gre...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
9,"""Good chap"" theory - Wikiquote","Blick, Andrew (2019) . ""Good Chaps No More"". S...",https://en.wikiquote.org/wiki/%22Good_chap%22_...


In [69]:
assert len(df) >= 1000

print("Requirement satisfied!")
print("Final number of rows:", len(df))

Requirement satisfied!
Final number of rows: 1258


In [70]:
df.to_csv(
    "quotes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("quotes.csv saved successfully!")

quotes.csv saved successfully!


In [71]:
import os

print("File exists:", os.path.exists("quotes.csv"))
print("File size:", os.path.getsize("quotes.csv"), "bytes")

File exists: True
File size: 454037 bytes


In [72]:
df_check = pd.read_csv("quotes.csv")

print("Rows in CSV:", len(df_check))
print("Columns:", df_check.columns.tolist())

df_check.head(10)

Rows in CSV: 1258
Columns: ['source_page', 'quote', 'url']


,source_page,quote,url
0,Ezra Klein - Wikiquote,If you are trying to create things that are go...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
1,Ezra Klein - Wikiquote,https://conversationswithtyler.com/episodes/ez...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
2,Ezra Klein - Wikiquote,There is no more profound human bias than the ...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
3,Ezra Klein - Wikiquote,"This Changes Everything (March 13, 2023), The ...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
4,Ezra Klein - Wikiquote,I have called the Republican Party an engine f...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
5,Ezra Klein - Wikiquote,Transcript: Ezra Klein Interviews Sam Rosenfel...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
6,Ezra Klein - Wikiquote,The Democratic Party does not need to choose t...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
7,Ezra Klein - Wikiquote,""" This Is the Way You Beat Trump — and Trumpis...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
8,"""Good chap"" theory - Wikiquote",The nature of its constitution tells you a gre...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
9,"""Good chap"" theory - Wikiquote","Blick, Andrew (2019) . ""Good Chaps No More"". S...",https://en.wikiquote.org/wiki/%22Good_chap%22_...


In [73]:
# Remove empty quotes
df_clean = df[df["quote"].notna()].copy()

# Remove quotes that are actually URLs
df_clean = df_clean[
    ~df_clean["quote"].str.strip().str.startswith(("http://", "https://"))
]

# Remove very short entries
df_clean = df_clean[
    df_clean["quote"].str.len() >= 30
]

# Remove duplicate quotes
df_clean = df_clean.drop_duplicates(subset=["quote"])

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Rows after cleaning:", len(df_clean))

Rows after cleaning: 1102


In [74]:
df_clean.head(20)

,source_page,quote,url
0,Ezra Klein - Wikiquote,If you are trying to create things that are go...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
1,Ezra Klein - Wikiquote,There is no more profound human bias than the ...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
2,Ezra Klein - Wikiquote,"This Changes Everything (March 13, 2023), The ...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
3,Ezra Klein - Wikiquote,I have called the Republican Party an engine f...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
4,Ezra Klein - Wikiquote,Transcript: Ezra Klein Interviews Sam Rosenfel...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
5,Ezra Klein - Wikiquote,The Democratic Party does not need to choose t...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
6,Ezra Klein - Wikiquote,""" This Is the Way You Beat Trump — and Trumpis...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
7,"""Good chap"" theory - Wikiquote",The nature of its constitution tells you a gre...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
8,"""Good chap"" theory - Wikiquote","Blick, Andrew (2019) . ""Good Chaps No More"". S...",https://en.wikiquote.org/wiki/%22Good_chap%22_...
9,"""Good chap"" theory - Wikiquote",Margaret Thatcher's former press secretary Sir...,https://en.wikiquote.org/wiki/%22Good_chap%22_...


In [75]:
print(df_clean["quote"].tolist()[:10])

['If you are trying to create things that are going to be continuously relevant three months from now, eight months from now, a year from now, five years from now, putting more emphasis on the history of a conflict like in Syria, is going to give you a better chance of having something that can be profitably rediscovered going forward. https://conversationswithtyler.com/episodes/ezra-klein/', 'There is no more profound human bias than the expectation that tomorrow will be like today. It is a powerful heuristic tool because it is almost always correct. This Changes Everything (March 13, 2023), The New York Times', 'This Changes Everything (March 13, 2023), The New York Times', 'I have called the Republican Party an engine for turning social resentment into tax cuts. Transcript: Ezra Klein Interviews Sam Rosenfeld and Daniel Schlozman (June 4, 2024), The Ezra Klein Show', 'Transcript: Ezra Klein Interviews Sam Rosenfeld and Daniel Schlozman (June 4, 2024), The Ezra Klein Show', 'The Demo

In [76]:
print("Number of cleaned rows:", len(df_clean))

for i, quote in enumerate(df_clean["quote"].head(30), start=1):
    print(f"{i}. {quote}")
    print("-" * 80)

Number of cleaned rows: 1102
1. If you are trying to create things that are going to be continuously relevant three months from now, eight months from now, a year from now, five years from now, putting more emphasis on the history of a conflict like in Syria, is going to give you a better chance of having something that can be profitably rediscovered going forward. https://conversationswithtyler.com/episodes/ezra-klein/
--------------------------------------------------------------------------------
2. There is no more profound human bias than the expectation that tomorrow will be like today. It is a powerful heuristic tool because it is almost always correct. This Changes Everything (March 13, 2023), The New York Times
--------------------------------------------------------------------------------
3. This Changes Everything (March 13, 2023), The New York Times
--------------------------------------------------------------------------------
4. I have called the Republican Party an eng

In [77]:
import re

df_final = df_clean.copy()

# Remove entries that look like references/sources rather than quotations
reference_patterns = [
    r"^This Changes Everything\s*\(",
    r"^Transcript:",
    r"^Blick, Andrew\s*\(",
    r"^Chapter \d+",
    r"^NS is the third book",
    r"^Mark Landler\s*,",
    r"^AL TV\s*,",
    r"^I Love the 90's",
    r"^Replying to a fan",
    r"^Ask Al",
]

pattern = "|".join(reference_patterns)

df_final = df_final[
    ~df_final["quote"].str.strip().str.match(pattern, case=False, na=False)
]

# Remove duplicate quotations
df_final = df_final.drop_duplicates(subset=["quote"])

# Reset index
df_final = df_final.reset_index(drop=True)

print("Final rows:", len(df_final))
print("Columns:", df_final.columns.tolist())

Final rows: 1092
Columns: ['source_page', 'quote', 'url']


In [78]:
df_final.head(20)

,source_page,quote,url
0,Ezra Klein - Wikiquote,If you are trying to create things that are go...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
1,Ezra Klein - Wikiquote,There is no more profound human bias than the ...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
2,Ezra Klein - Wikiquote,I have called the Republican Party an engine f...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
3,Ezra Klein - Wikiquote,The Democratic Party does not need to choose t...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
4,Ezra Klein - Wikiquote,""" This Is the Way You Beat Trump — and Trumpis...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
5,"""Good chap"" theory - Wikiquote",The nature of its constitution tells you a gre...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
6,"""Good chap"" theory - Wikiquote",Margaret Thatcher's former press secretary Sir...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
7,"""Good chap"" theory - Wikiquote","""The theory of the ""good chap"" , BBC News (17 ...",https://en.wikiquote.org/wiki/%22Good_chap%22_...
8,"""Good chap"" theory - Wikiquote",But the real “trouble” lies in the consequence...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
9,"""Good chap"" theory - Wikiquote","""Peter Hennessy: Boris Johnson has killed off ...",https://en.wikiquote.org/wiki/%22Good_chap%22_...


In [79]:
df_final.to_csv(
    "quotes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final CSV saved successfully!")
print("Number of rows:", len(df_final))


Final CSV saved successfully!
Number of rows: 1092


In [80]:
final_check = pd.read_csv("quotes.csv")

print("Rows in CSV:", len(final_check))
print("Columns:", final_check.columns.tolist())
print("Missing values:")
print(final_check.isnull().sum())

Rows in CSV: 1092
Columns: ['source_page', 'quote', 'url']
Missing values:
source_page    0
quote          0
url            0
dtype: int64


In [81]:
final_check.head(10)


,source_page,quote,url
0,Ezra Klein - Wikiquote,If you are trying to create things that are go...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
1,Ezra Klein - Wikiquote,There is no more profound human bias than the ...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
2,Ezra Klein - Wikiquote,I have called the Republican Party an engine f...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
3,Ezra Klein - Wikiquote,The Democratic Party does not need to choose t...,https://en.wikiquote.org/wiki/%22Ezra_Klein%22
4,Ezra Klein - Wikiquote,""" This Is the Way You Beat Trump — and Trumpis...",https://en.wikiquote.org/wiki/%22Ezra_Klein%22
5,"""Good chap"" theory - Wikiquote",The nature of its constitution tells you a gre...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
6,"""Good chap"" theory - Wikiquote",Margaret Thatcher's former press secretary Sir...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
7,"""Good chap"" theory - Wikiquote","""The theory of the ""good chap"" , BBC News (17 ...",https://en.wikiquote.org/wiki/%22Good_chap%22_...
8,"""Good chap"" theory - Wikiquote",But the real “trouble” lies in the consequence...,https://en.wikiquote.org/wiki/%22Good_chap%22_...
9,"""Good chap"" theory - Wikiquote","""Peter Hennessy: Boris Johnson has killed off ...",https://en.wikiquote.org/wiki/%22Good_chap%22_...


In [82]:
from google.colab import files

files.download("quotes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>